In [ ]:
# ============================================================
# QUANTPROBE: HARDWARE-EFFICIENCY BENCHMARK
# Configuration: Qwen2.5-1.5B-Instruct, FP16, Tesla T4
# Measures: Model loading, latency, throughput, GPU memory
# ============================================================

import os
import json
import time
import csv
import platform
import statistics
from pathlib import Path
from datetime import datetime, timezone

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
PRECISION = "FP16"

MAX_NEW_TOKENS = 128
WARMUP_RUNS = 2
MEASURED_RUNS = 5

OUTPUT_DIR = Path("/kaggle/working/hardware_efficiency")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU unavailable. Enable GPU acceleration in Kaggle."
    )

DEVICE_INDEX = 0
torch.cuda.set_device(DEVICE_INDEX)
DEVICE = torch.device(f"cuda:{DEVICE_INDEX}")

print("GPU:", torch.cuda.get_device_name(DEVICE_INDEX))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

# ------------------------------------------------------------
# 2. Benchmark prompt
# ------------------------------------------------------------

PROMPT = (
    "Answer in one concise paragraph: What is quantization "
    "in a language model, and why can it affect factual reliability?"
)

# ------------------------------------------------------------
# 3. Load tokenizer and model
# ------------------------------------------------------------

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(DEVICE)

load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": DEVICE_INDEX}
)

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_load_seconds = time.perf_counter() - load_start

print(f"\nModel loading time: {model_load_seconds:.3f} seconds")

# ------------------------------------------------------------
# 4. Prepare input
# ------------------------------------------------------------

inputs = tokenizer(
    PROMPT,
    return_tensors="pt"
).to(DEVICE)

input_tokens = inputs["input_ids"].shape[1]

# ------------------------------------------------------------
# 5. Generation function
# ------------------------------------------------------------

def generate_once():
    with torch.inference_mode():
        return model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id
        )

# ------------------------------------------------------------
# 6. Warm-up runs
# ------------------------------------------------------------

print("\nRunning warm-up iterations...")

for _ in range(WARMUP_RUNS):
    _ = generate_once()

torch.cuda.synchronize(DEVICE)

# Reset memory peaks after warm-up
torch.cuda.reset_peak_memory_stats(DEVICE)

# ------------------------------------------------------------
# 7. Measured runs
# ------------------------------------------------------------

run_results = []

print("Running measured iterations...")

for run_id in range(1, MEASURED_RUNS + 1):

    torch.cuda.synchronize(DEVICE)
    start_time = time.perf_counter()

    output_ids = generate_once()

    torch.cuda.synchronize(DEVICE)
    elapsed_seconds = time.perf_counter() - start_time

    total_tokens = output_ids.shape[1]
    generated_tokens = max(0, total_tokens - input_tokens)

    throughput = (
        generated_tokens / elapsed_seconds
        if elapsed_seconds > 0 else None
    )

    run_results.append({
        "run": run_id,
        "latency_seconds": elapsed_seconds,
        "input_tokens": int(input_tokens),
        "generated_tokens": int(generated_tokens),
        "tokens_per_second": throughput
    })

    print(
        f"Run {run_id}: "
        f"latency={elapsed_seconds:.3f}s, "
        f"tokens={generated_tokens}, "
        f"throughput={throughput:.2f} tokens/s"
    )

# ------------------------------------------------------------
# 8. Calculate hardware metrics
# ------------------------------------------------------------

latencies = [
    row["latency_seconds"]
    for row in run_results
]

throughputs = [
    row["tokens_per_second"]
    for row in run_results
    if row["tokens_per_second"] is not None
]

mean_latency = statistics.mean(latencies)

mean_throughput = (
    statistics.mean(throughputs)
    if throughputs else None
)

latency_std = (
    statistics.stdev(latencies)
    if len(latencies) > 1 else 0.0
)

peak_allocated_gib = (
    torch.cuda.max_memory_allocated(DEVICE)
    / (1024 ** 3)
)

peak_reserved_gib = (
    torch.cuda.max_memory_reserved(DEVICE)
    / (1024 ** 3)
)

gpu_properties = torch.cuda.get_device_properties(DEVICE_INDEX)

gpu_name = torch.cuda.get_device_name(DEVICE_INDEX)

gpu_total_memory_gib = (
    gpu_properties.total_memory / (1024 ** 3)
)

# ------------------------------------------------------------
# 9. Assemble results
# ------------------------------------------------------------

results = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),

    "status": "FP16 measured; INT8 and INT4 pending",

    "model": MODEL_ID,
    "precision": PRECISION,

    "hardware": {
        "gpu_name": gpu_name,
        "gpu_total_memory_gib": gpu_total_memory_gib,
        "cuda_version": torch.version.cuda
    },

    "software": {
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "transformers_version": transformers.__version__
    },

    "settings": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "warmup_runs": WARMUP_RUNS,
        "measured_runs": MEASURED_RUNS,
        "do_sample": False,
        "num_beams": 1,
        "dtype": "float16",
        "input_tokens": int(input_tokens),
        "prompt": PROMPT
    },

    "model_load_seconds": model_load_seconds,

    "mean_latency_seconds": mean_latency,

    "latency_std_seconds": latency_std,

    "mean_tokens_per_second": mean_throughput,

    "peak_allocated_memory_gib": peak_allocated_gib,

    "peak_reserved_memory_gib": peak_reserved_gib,

    "individual_runs": run_results,

    "limitations": [
        "Only FP16 was measured successfully.",
        "INT8 and INT4 measurements are pending.",
        "Results may vary across hardware and runtime versions.",
        "This benchmark uses greedy decoding and is separate "
        "from the main experiment's sampling configuration."
    ]
}

# ------------------------------------------------------------
# 10. Save JSON output
# ------------------------------------------------------------

json_path = (
    OUTPUT_DIR / "hardware_efficiency_fp16_results.json"
)

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

# ------------------------------------------------------------
# 11. Save individual runs as CSV
# ------------------------------------------------------------

csv_path = (
    OUTPUT_DIR / "hardware_efficiency_fp16_runs.csv"
)

with open(csv_path, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=list(run_results[0].keys())
    )

    writer.writeheader()
    writer.writerows(run_results)

# ------------------------------------------------------------
# 12. Print summary
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("QUANTPROBE HARDWARE-EFFICIENCY RESULTS")
print("=" * 55)

print(f"Model:                  {MODEL_ID}")
print(f"Precision:              {PRECISION}")
print(f"GPU:                    {gpu_name}")
print(f"Model loading time:     {model_load_seconds:.3f} seconds")
print(f"Mean latency:           {mean_latency:.3f} seconds")
print(f"Mean throughput:        {mean_throughput:.2f} tokens/sec")
print(f"Latency standard dev.:  {latency_std:.3f} seconds")
print(f"Peak allocated memory:  {peak_allocated_gib:.3f} GiB")
print(f"Peak reserved memory:   {peak_reserved_gib:.3f} GiB")

print("\nSaved output files:")
print(json_path)
print(csv_path)

print("\nINT8 and INT4 remain pending.")
print("=" * 55)

In [ ]:
"""
QuantProbe 2 evaluation script (v9).

Changes from v8:
  - write_excel() writes two additional sheets: MultiSeed_Summary and Per_Seed.
    The existing Summary sheet still shows the primary-seed metrics.
  - Threshold sensitivity is computed per (model, precision), not pooled.
    Output sheets: Threshold_Sensitivity (raw) and Threshold_Sensitivity_Rows
    (flat, one row per model/precision/threshold).

Everything else is unchanged from v8.

Requirements:
    pip install bitsandbytes>=0.46.1 accelerate>=0.26.0 scipy>=1.11.0
    pip install transformers torch sentencepiece openpyxl tqdm

Data files:
    ./data/TruthfulQA.csv
    ./data/freshqa.csv          (must contain a date column)
    ./data/QuantHallbench.json  (every record must contain "split":"dev"|"test")
"""

# ---------------------------------------------------------------------------
# Dependency check
# ---------------------------------------------------------------------------
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for _pkg, _min in [("bitsandbytes", "0.46.1"), ("accelerate", "0.26.0"), ("scipy", "1.11.0")]:
    try:
        mod = __import__(_pkg)
        from packaging.version import Version
        if Version(getattr(mod, "__version__", "0")) < Version(_min):
            raise ImportError
    except Exception:
        print(f"Installing {_pkg}>={_min} ...")
        _pip(f"{_pkg}>={_min}")

try:
    from tqdm import tqdm
except ImportError:
    _pip("tqdm")
    from tqdm import tqdm

# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import gc, json, math, time, logging, random, hashlib
from pathlib import Path
from datetime import datetime
from math import comb

# ---------------------------------------------------------------------------
# Third-party
# ---------------------------------------------------------------------------
import torch
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig,
)
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
RUN_MODE = "pilot"
SINGLE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

DATASET_BASE = "./data"
KAGGLE_BASE = "/kaggle/input/datasets/samuelstephen77/quanthall-dataset"

TRUTHFULQA_PATH = f"{DATASET_BASE}/TruthfulQA.csv"
FRESHQA_PATH    = f"{DATASET_BASE}/freshqa.csv"
BENCHJSON_PATH  = "/kaggle/working/QuantHallbench_withsplit.json"


MAX_NEW_TOKENS = 100
TEMPERATURE    = 0.1
TOP_P          = 1.0
TOP_K          = 0
DO_SAMPLE      = True
NUM_BEAMS      = 1
ECE_BINS       = 10
SAVE_EVERY     = 25

PILOT_N         = 30
SAMPLE_CAP_FP16 = 500
SAMPLE_CAP_8BIT = 500
SAMPLE_CAP_4BIT = 500
SAMPLE_CAP_BY_BITS = {16: 500, 8: 500, 4: 500}

CALIB_FRACTION = 0.20

RANDOM_SEEDS = [42, 123, 7]
RANDOM_SEED  = RANDOM_SEEDS[0]

QHS_ALPHA = 0.40
QHS_BETA  = 0.25
QHS_GAMMA = 0.20
QHS_DELTA = 0.15

THRESHOLD_GRID = [0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20]

MODEL_CUTOFFS = {
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0": "2023-12-31",
    "Qwen/Qwen2.5-1.5B-Instruct":         "2024-06-30",
    "microsoft/phi-2":                    "2023-06-30",
}
FRESHQA_DEFAULT_CUTOFF = "2023-12-31"

RESULTS_DIR    = Path("results_v9")
CHECKPOINT_DIR = Path("checkpoints_v9")

ALL_EXPERIMENTS = [
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", 16),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",  8),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",  4),
    ("Qwen/Qwen2.5-1.5B-Instruct",         16),
    ("Qwen/Qwen2.5-1.5B-Instruct",          8),
    ("Qwen/Qwen2.5-1.5B-Instruct",          4),
    ("microsoft/phi-2",                    16),
    ("microsoft/phi-2",                     8),
    ("microsoft/phi-2",                     4),
]

ABSTENTION_MARKERS = [
    "i don't know", "i do not know", "i cannot determine",
    "i'm not sure", "i am not sure", "i cannot answer",
    "i don't have information", "i am unable to",
    "this is unknown", "cannot be determined", "i'm unable to",
    "i have no information", "i cannot say", "it is not known",
]

FACTUALITY_VALIDATION_STATUS = "unvalidated_heuristic"

logging.basicConfig(format="%(asctime)s  %(levelname)-8s  %(message)s",
                    level=logging.INFO, datefmt="%H:%M:%S")
log = logging.getLogger("qp2")

import warnings
warnings.filterwarnings("ignore", message=".*cast from torch.bfloat16 to float16.*")
warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")

RESULTS_DIR.mkdir(exist_ok=True, parents=True)
CHECKPOINT_DIR.mkdir(exist_ok=True, parents=True)


# ---------------------------------------------------------------------------
# Experiment selection
# ---------------------------------------------------------------------------
def get_experiments():
    if RUN_MODE in ("pilot", "full"):
        return ALL_EXPERIMENTS
    if RUN_MODE == "single":
        return [(m, b) for m, b in ALL_EXPERIMENTS if m == SINGLE_MODEL]
    if RUN_MODE == "smoke":
        return [("TinyLlama/TinyLlama-1.1B-Chat-v1.0", 16)]
    return []


def get_sample_size(bit_width):
    if RUN_MODE in ("pilot", "smoke"):
        return PILOT_N
    return SAMPLE_CAP_BY_BITS.get(bit_width, SAMPLE_CAP_FP16)


# ---------------------------------------------------------------------------
# Statistical primitives
# ---------------------------------------------------------------------------
def _auroc(scores, labels):
    pos = [s for s, l in zip(scores, labels) if l == 1]
    neg = [s for s, l in zip(scores, labels) if l == 0]
    if not pos or not neg:
        return None
    wins = sum(1 for p in pos for n in neg if p > n) + 0.5 * sum(1 for p in pos for n in neg if p == n)
    return wins / (len(pos) * len(neg))


def _risk_coverage_curve(confidences, correct_flags):
    if not confidences:
        return [], [], 0.0
    order = np.argsort(-np.array(confidences))
    c = np.array(correct_flags)[order]
    covs, risks = [], []
    for k in range(1, len(c) + 1):
        covs.append(k / len(c))
        risks.append(1.0 - c[:k].mean())
    aurc = float(np.trapz(risks, covs)) if len(covs) > 1 else 0.0
    return covs, risks, aurc


def _ece_proxy(confidences, correct_flags, bins=ECE_BINS):
    if not confidences:
        return 0.0
    conf = np.array(confidences); acc = np.array(correct_flags, dtype=float)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        m = (conf >= lo) & (conf < hi) if i < bins - 1 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(conf)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def _brier(conf, y):
    if not conf:
        return 0.0
    return float(np.mean([(c - yy) ** 2 for c, yy in zip(conf, y)]))


def _mcnemar_exact_p(k01, k10):
    disc = k01 + k10
    if disc == 0:
        return 1.0
    k = min(k01, k10)
    tail = sum(comb(disc, i) for i in range(0, k + 1)) / (2 ** disc)
    return min(1.0, 2.0 * tail)


def _holm_correct(p_values):
    m = len(p_values)
    if m == 0:
        return []
    order = np.argsort(p_values)
    adjusted = np.ones(m)
    prev = 0.0
    for rank, idx in enumerate(order):
        adj = min(1.0, (m - rank) * p_values[idx])
        adjusted[idx] = max(adj, prev)
        prev = adjusted[idx]
    return adjusted


def _paired_bootstrap_ci(a, b, n_boot=2000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    a = np.asarray(a, float); b = np.asarray(b, float)
    if a.shape != b.shape or len(a) == 0:
        return {"mean_diff": None, "ci_lo": None, "ci_hi": None, "n_pairs": 0}
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(a), len(a))
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    return {"mean_diff": float(a.mean() - b.mean()),
            "ci_lo": float(lo), "ci_hi": float(hi),
            "n_pairs": int(len(a)), "n_boot": n_boot}


# ---------------------------------------------------------------------------
# Self-tests
# ---------------------------------------------------------------------------
def _selftest_primitives():
    assert _auroc([3, 4, 5], [0, 0, 0]) is None
    assert _auroc([3, 4, 5], [1, 1, 1]) is None
    assert _auroc([2, 3, 4, 5], [0, 0, 1, 1]) == 1.0
    assert _auroc([4, 3, 2, 1], [0, 0, 1, 1]) == 0.0
    assert abs(_auroc([1, 1, 1, 1], [0, 0, 1, 1]) - 0.5) < 1e-9
    assert abs(_ece_proxy([1.0, 1.0], [1.0, 1.0])) < 1e-6
    assert abs(_brier([1.0, 0.0], [1.0, 0.0])) < 1e-9
    expected = 2 * sum(comb(12, i) for i in range(3)) / (2 ** 12)
    assert abs(_mcnemar_exact_p(10, 2) - expected) < 1e-9
    assert _mcnemar_exact_p(0, 0) == 1.0
    adj = _holm_correct([0.001, 0.02, 0.5, 0.9])
    assert all(0 <= a <= 1 for a in adj)
    assert adj[0] <= adj[1] <= adj[2] <= adj[3]
    ci = _paired_bootstrap_ci([1, 2, 3], [1, 2, 3])
    assert abs(ci["mean_diff"]) < 1e-9
    _, _, aurc = _risk_coverage_curve([0.9, 0.8, 0.7, 0.6], [1, 1, 1, 1])
    assert aurc < 0.01
    log.info("  [ok] statistical primitives")


def _selftest_entropy():
    logits = torch.zeros(1, 100)
    got = _compute_entropy([logits])
    expected = math.log(100)
    assert abs(got - expected) < 1e-3
    log.info(f"  [ok] entropy computation ({got:.4f})")


def _selftest_heuristic_classifier():
    assert classify_heuristic("I don't know", "Paris") == "abstained"
    assert classify_heuristic("The capital is Paris.", "Paris") == "correct"
    assert classify_heuristic("The capital is London.", "Paris") == "incorrect"
    assert classify_heuristic("206 bones", "206") == "correct"
    log.info("  [ok] heuristic classifier")


def _selftest_metrics():
    recs = [{"heuristic_label": "correct", "entropy": 0.02, "uncertainty_label": "low"} for _ in range(5)]
    recs += [{"heuristic_label": "incorrect", "entropy": 0.10, "uncertainty_label": "high"} for _ in range(5)]
    m = compute_metrics(recs)
    assert m["error_rate"] == 0.5
    assert m["accuracy"] == 0.5
    assert 0 <= m["crs_clamped"] <= 1.0
    log.info("  [ok] compute_metrics")


def _selftest_transition():
    a = [{"dataset": "D", "item_id": "1", "heuristic_label": "correct"},
         {"dataset": "D", "item_id": "2", "heuristic_label": "incorrect", "uncertainty_label": "low"},
         {"dataset": "D", "item_id": "3", "heuristic_label": "incorrect", "uncertainty_label": "high"}]
    b = [{"dataset": "D", "item_id": "1", "heuristic_label": "incorrect", "uncertainty_label": "high"},
         {"dataset": "D", "item_id": "2", "heuristic_label": "incorrect", "uncertainty_label": "high"},
         {"dataset": "D", "item_id": "3", "heuristic_label": "incorrect", "uncertainty_label": "high"}]
    out = paired_transition_matrix(a, b)
    assert out["n_common"] == 3
    assert out["FP16_correct__quant_incorrect"] == 1
    assert out["FP16_incorrect__quant_correct"] == 0
    log.info("  [ok] paired_transition_matrix")


def _selftest_sensitivity():
    recs = [{"heuristic_label": "incorrect", "entropy": e, "uncertainty_label": "low"}
            for e in [0.001, 0.02, 0.04, 0.06, 0.10, 0.15]]
    df = threshold_sensitivity(recs)
    assert len(df) == len(THRESHOLD_GRID)
    assert df["error_rate"].iloc[0] == 1.0
    assert abs(df["ocs_proxy"].iloc[0] - 1 / 6) < 1e-9
    log.info("  [ok] threshold_sensitivity")


def _selftest_calibration():
    recs = [{"item_id": f"c{i}", "heuristic_label": "correct", "entropy": 0.03} for i in range(40)]
    recs += [{"item_id": f"e{i}", "heuristic_label": "incorrect", "entropy": 0.02 + 0.005 * i} for i in range(20)]
    theta, calib_ids, remaining, method = calibrate_threshold_heldout(recs, frac=0.3, seed=1)
    remaining_ids = {r["item_id"] for r in remaining}
    assert calib_ids.isdisjoint(remaining_ids)
    assert len(calib_ids) + len(remaining) == len(recs)
    assert method in ("held_out_median_of_errors",
                      "fallback_global_too_few_scored",
                      "fallback_global_too_few_errors")
    log.info(f"  [ok] calibration (theta={theta:.4f}, method={method})")


def _selftest_checkpoint_prefix():
    ds = "TestDS"
    model = "TestModel/x"
    bit = 16
    seed = 42

    dataset_ids = ["a", "b", "c", "d", "e"]

    if _ckpt_path(ds, model, bit, seed).exists():
        _ckpt_path(ds, model, bit, seed).unlink()
    assert load_checkpoint(ds, model, bit, seed, dataset_ids) == []

    recs_valid = [{"item_id": "a"}, {"item_id": "b"}]
    save_checkpoint(recs_valid, ds, model, bit, seed, dataset_ids)
    resumed = load_checkpoint(ds, model, bit, seed, dataset_ids)
    assert [r["item_id"] for r in resumed] == ["a", "b"], "valid prefix should resume"

    recs_bad = [{"item_id": "a"}, {"item_id": "x"}]
    save_checkpoint(recs_bad, ds, model, bit, seed, dataset_ids)
    assert load_checkpoint(ds, model, bit, seed, dataset_ids) == [], \
        "mismatched prefix must be rejected"

    _ckpt_path(ds, model, bit, seed).unlink()
    log.info("  [ok] checkpoint prefix validation")


def _run_all_selftests():
    log.info("=" * 60)
    log.info("SELF-TESTS")
    log.info("=" * 60)
    _selftest_primitives()
    _selftest_entropy()
    _selftest_heuristic_classifier()
    _selftest_metrics()
    _selftest_transition()
    _selftest_sensitivity()
    _selftest_calibration()
    _selftest_checkpoint_prefix()
    log.info("=" * 60)
    log.info("ALL SELF-TESTS PASSED")
    log.info("=" * 60)


# ---------------------------------------------------------------------------
# Entropy and generation
# ---------------------------------------------------------------------------
def _compute_entropy(scores):
    if not scores:
        return 0.0
    ent = []
    for step in scores:
        logits = step[0]
        lp = torch.log_softmax(logits, dim=-1)
        p = lp.exp()
        ent.append(-(p * lp).sum().item())
    return float(np.mean(ent))


def generate_response(prompt, tok, model):
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=512, padding=True).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
            top_p=TOP_P, top_k=TOP_K, do_sample=DO_SAMPLE, num_beams=NUM_BEAMS,
            output_scores=True, return_dict_in_generate=True,
            pad_token_id=tok.eos_token_id)
    latency = time.time() - t0
    gen_ids = out.sequences[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    return text, _compute_entropy(out.scores), latency


# ---------------------------------------------------------------------------
# Heuristic classifier
# ---------------------------------------------------------------------------
_STOPWORDS = {
    "the", "and", "for", "was", "were", "that", "this", "with", "from", "have",
    "been", "are", "its", "per", "what", "when", "where", "who", "why", "how",
    "which", "can", "could", "would", "should", "does", "did", "will", "than",
    "more", "much", "most", "less", "also", "into", "onto", "upon", "gets",
}


def classify_heuristic(response, reference):
    import re
    r = response.lower().strip()
    if any(m in r for m in ABSTENTION_MARKERS):
        return "abstained"
    ref = str(reference or "").lower().strip()
    if not ref:
        return "unscored"
    if ref in r:
        return "correct"
    toks = [t for t in re.findall(r"[a-z]+", ref) if len(t) > 3 and t not in _STOPWORDS]
    if toks:
        m = [t for t in toks if t in r]
        if len(m) / len(toks) >= 0.6 and len(m) >= 2:
            return "correct"
    nums = re.findall(r"\d+", ref)
    if nums and all(n in r for n in nums):
        return "correct"
    return "incorrect"


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def compute_metrics(results):
    n = len(results)
    if n == 0:
        return {"n": 0}
    labels = [r.get("heuristic_label", "unscored") for r in results]
    elig = [r for r in results if r.get("heuristic_label") in ("correct", "incorrect")]
    incorrect = [r for r in elig if r.get("heuristic_label") == "incorrect"]
    n_elig = len(elig)
    abstained = sum(1 for l in labels if l == "abstained")
    correct_n = sum(1 for l in labels if l == "correct")

    err_rate = len(incorrect) / n_elig if n_elig else 0.0
    acc = correct_n / n_elig if n_elig else 0.0
    abst = abstained / n

    err_ent = [float(r.get("entropy", 0.0)) for r in incorrect]
    low_u = sum(1 for r in incorrect if r.get("uncertainty_label") == "low")
    hr_proxy = (len(incorrect) - low_u) / n_elig if n_elig else 0.0
    ocs_proxy = low_u / n_elig if n_elig else 0.0

    conf = [min(1.0, max(0.0, math.exp(-float(r.get("entropy", 0.0))))) for r in elig]
    y = [1.0 if r["heuristic_label"] == "correct" else 0.0 for r in elig]
    ece = _ece_proxy(conf, y)
    brier = _brier(conf, y)
    auroc_raw = _auroc([-c for c in conf], [1 - int(yy) for yy in y])
    auroc = auroc_raw if auroc_raw is not None else float("nan")
    _, _, aurc = _risk_coverage_curve(conf, y)

    raw = 1 - QHS_ALPHA * hr_proxy - QHS_BETA * ocs_proxy - QHS_GAMMA * ece + QHS_DELTA * abst
    crs = float(min(1.0, max(0.0, raw)))

    return {
        "accuracy": round(acc, 4),
        "error_rate": round(err_rate, 4),
        "hallucination_rate_proxy": round(hr_proxy, 4),
        "overconfidence_proxy": round(ocs_proxy, 4),
        "abstention_rate": round(abst, 4),
        "err_entropy_mean": round(float(np.mean(err_ent)), 4) if err_ent else 0.0,
        "err_entropy_median": round(float(np.median(err_ent)), 4) if err_ent else 0.0,
        "err_entropy_std": round(float(np.std(err_ent)), 4) if err_ent else 0.0,
        "err_entropy_values": err_ent,
        "ece_proxy": round(ece, 4),
        "brier": round(brier, 4),
        "auroc_error": (round(auroc, 4) if not math.isnan(auroc) else None),
        "aurc": round(aurc, 4),
        "crs_clamped": round(crs, 4),
        "n": n,
        "n_scored": n_elig,
        "factuality_validation_status": FACTUALITY_VALIDATION_STATUS,
    }


# ---------------------------------------------------------------------------
# Transition and sensitivity
# ---------------------------------------------------------------------------
def _bucket(r):
    lab = r.get("heuristic_label")
    if lab == "correct":
        return "correct"
    if lab == "abstained":
        return "abstained"
    if lab == "incorrect":
        return "confident_error" if r.get("uncertainty_label") == "low" else "uncertain_error"
    return "unscored"


def paired_transition_matrix(recs_a, recs_b):
    key = lambda r: (r.get("dataset"), r.get("item_id"))
    a_map = {key(r): r for r in recs_a}
    b_map = {key(r): r for r in recs_b}
    common = sorted(set(a_map) & set(b_map))
    buckets = ["correct", "confident_error", "uncertain_error", "abstained", "unscored"]
    mat = pd.DataFrame(0, index=buckets, columns=buckets, dtype=int)
    k01 = k10 = 0
    for k in common:
        la, lb = a_map[k].get("heuristic_label"), b_map[k].get("heuristic_label")
        mat.loc[_bucket(a_map[k]), _bucket(b_map[k])] += 1
        if la == "correct" and lb == "incorrect":
            k01 += 1
        if la == "incorrect" and lb == "correct":
            k10 += 1
    return {"n_common": len(common), "matrix": mat,
            "FP16_correct__quant_incorrect": k01,
            "FP16_incorrect__quant_correct": k10,
            "mcnemar_exact_p": _mcnemar_exact_p(k01, k10)}


def threshold_sensitivity(results, grid=THRESHOLD_GRID):
    rows = []
    elig = [r for r in results if r.get("heuristic_label") in ("correct", "incorrect")]
    if not elig:
        return pd.DataFrame()
    for th in grid:
        incorrect = [r for r in elig if r.get("heuristic_label") == "incorrect"]
        low_u = sum(1 for r in incorrect if float(r.get("entropy", 0)) < th)
        rows.append({
            "threshold": th,
            "n_scored": len(elig),
            "error_rate": len(incorrect) / len(elig),
            "ocs_proxy": low_u / len(elig),
            "hr_proxy": (len(incorrect) - low_u) / len(elig),
        })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Threshold calibration (held out)
# ---------------------------------------------------------------------------
def calibrate_threshold_heldout(records, frac=CALIB_FRACTION, seed=RANDOM_SEED):
    rng = random.Random(seed)
    scored = [r for r in records if r.get("heuristic_label") in ("correct", "incorrect")]
    if len(scored) < 20:
        log.warning(f"  Too few scored items ({len(scored)}) - calibration skipped.")
        return 0.05, set(), records, "fallback_global_too_few_scored"
    rng.shuffle(scored)
    n_cal = max(5, int(len(scored) * frac))
    calib = scored[:n_cal]
    calib_ids = {r["item_id"] for r in calib if "item_id" in r}
    err_ent = [float(r.get("entropy", 0.0)) for r in calib if r.get("heuristic_label") == "incorrect"]
    if len(err_ent) < 3:
        log.warning(f"  Only {len(err_ent)} errors in calibration slice - skipped.")
        return 0.05, set(), records, "fallback_global_too_few_errors"
    theta = float(np.median(err_ent))
    remaining = [r for r in records if r.get("item_id") not in calib_ids]
    log.info(f"  Calibrated theta={theta:.4f} from {len(err_ent)} errors "
             f"on {n_cal}-item slice; {len(remaining)} items remain for evaluation.")
    return theta, calib_ids, remaining, "held_out_median_of_errors"


# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------
def _bnb_config_dict(bit_width):
    if bit_width == 4:
        return {"load_in_4bit": True, "bnb_4bit_quant_type": "nf4",
                "bnb_4bit_compute_dtype": "float16",
                "bnb_4bit_quant_storage": "float16"}
    if bit_width == 8:
        return {"load_in_8bit": True,
                "llm_int8_enable_fp32_cpu_offload": False,
                "llm_int8_threshold": 6.0}
    return {"precision": "fp16"}


def load_model(model_name, bit_width):
    log.info(f"Loading {model_name} at {bit_width}-bit ...")
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    bnb = None
    if bit_width == 4:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_quant_storage=torch.float16)
    elif bit_width == 8:
        bnb = BitsAndBytesConfig(load_in_8bit=True,
                                 llm_int8_enable_fp32_cpu_offload=False,
                                 llm_int8_threshold=6.0)

    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    if not hasattr(cfg, "pad_token_id") or cfg.pad_token_id is None:
        cfg.pad_token_id = tok.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_name, config=cfg, quantization_config=bnb, device_map="auto",
        trust_remote_code=True, low_cpu_mem_usage=True, torch_dtype=torch.float16)
    model.eval()
    log.info(f"  Loaded. Config: {_bnb_config_dict(bit_width)}")
    return tok, model


def unload_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---------------------------------------------------------------------------
# Dataset loaders
# ---------------------------------------------------------------------------
def _resolve(path, kaggle_fallback):
    p = Path(path)
    if p.exists():
        return p
    k = Path(kaggle_fallback)
    if k.exists():
        return k
    raise FileNotFoundError(f"Neither {p} nor {k} exists")


def _frozen_ids(records, salt=""):
    for r in records:
        key = f"{r['dataset']}|{r.get('prompt','')[:80]}|{salt}".encode()
        r["item_id"] = hashlib.sha1(key).hexdigest()[:16]
    return records


def load_truthfulqa(n=None):
    path = _resolve(TRUTHFULQA_PATH, f"{KAGGLE_BASE}/TruthfulQA.csv")
    log.info(f"Loading TruthfulQA from {path} ...")
    df = pd.read_csv(path)
    recs = [{"dataset": "TruthfulQA",
             "prompt": f"Question: {row['Question']}\nAnswer:",
             "reference": str(row.get("Best Answer", "")),
             "category": "factual",
             "split": "test"} for _, row in df.iterrows()]
    recs = _frozen_ids(recs, salt="tqa")
    if n and n < len(recs):
        random.Random(RANDOM_SEED).shuffle(recs)
    log.info(f"  TruthfulQA: {len(recs)} records.")
    return recs[:n] if n else recs


def load_freshqa(n=None, model_name=None):
    path = _resolve(FRESHQA_PATH, f"{KAGGLE_BASE}/freshqa.csv")
    log.info(f"Loading FreshQA from {path} ...")
    # Row 1 is a warning, row 2 is the real header.
    df = pd.read_csv(path, skiprows=2)

    # FreshQA's date column is called 'effective_year' (values like
    # "before 2022", "2025", "2026"). We parse the year out of it.
    if "effective_year" not in df.columns:
        raise RuntimeError(
            "FreshQA: expected an 'effective_year' column. Found columns: "
            f"{list(df.columns)}"
        )
        
    cutoff = MODEL_CUTOFFS.get(model_name, FRESHQA_DEFAULT_CUTOFF)
    recs = []
    def _effective_year_to_date(value):
        """Parse 'before 2022' / '2025' / '2026' into a YYYY-01-01 string, or None."""
        s = str(value or "").strip().lower()
        if not s:
            return None
        # Extract the first 4-digit year in the string.
        import re as _re
        m = _re.search(r"(19|20)\d{2}", s)
        if not m:
            return None
        return f"{m.group(0)}-01-01"

    for _, row in df.iterrows():
        q = row.get("question", "")
        if pd.isna(q) or not str(q).strip():
            continue
        qd = _effective_year_to_date(row.get("effective_year", ""))
        if qd is None:
            continue
        if qd > cutoff:
            continue
        a = row.get("answer_0", "")
        recs.append({"dataset": "FreshQA",
                     "prompt": f"Question: {q}\nAnswer:",
                     "reference": "" if pd.isna(a) else str(a),
                     "category": "temporal",
                     "date": qd,
                     "split": "test"})
    recs = _frozen_ids(recs, salt=f"fqa|{cutoff}")
    log.info(f"  FreshQA: {len(recs)} records (cutoff={cutoff}).")
    if n and n < len(recs):
        random.Random(RANDOM_SEED).shuffle(recs)
    return recs[:n] if n else recs


def load_quanthallbench(n=None, split=None):
    path = _resolve(BENCHJSON_PATH, f"{KAGGLE_BASE}/QuantHallbench.json")
    log.info(f"Loading QuantProbe-Bench from {path} ...")
    with open(path) as f:
        records = json.load(f)
    for r in records:
        r["dataset"] = "QuantHall-Bench"
        if not r["prompt"].startswith("Question:"):
            r["prompt"] = f"Question: {r['prompt']}\nAnswer:"
        if "split" not in r or r["split"] not in ("dev", "test"):
            raise RuntimeError(
                "QuantProbe-Bench records must carry an explicit split field "
                "('dev' or 'test'). Add the field to QuantHallbench.json."
            )
    records = _frozen_ids(records, salt="qhb")
    if split is not None:
        records = [r for r in records if r["split"] == split]
    log.info(f"  QuantHall-Bench: {len(records)} records"
             + (f" (split={split})" if split else " (all splits)"))
    if n and n < len(records):
        random.Random(RANDOM_SEED).shuffle(records)
    return records[:n] if n else records


# ---------------------------------------------------------------------------
# Checkpoints
# ---------------------------------------------------------------------------
def _ckpt_path(ds, model, bit, seed):
    safe_m = model.replace("/", "_")
    safe_d = ds.replace(" ", "_")
    return CHECKPOINT_DIR / f"ckpt__{safe_m}__{safe_d}__{bit}bit__seed{seed}.json"


def _ckpt_meta(ds, model, bit, seed, dataset_ids):
    return {
        "dataset": ds,
        "model": model,
        "bit_width": bit,
        "seed": seed,
        "generation_params": {
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P, "top_k": TOP_K,
            "do_sample": DO_SAMPLE, "num_beams": NUM_BEAMS,
        },
        "bnb_config": _bnb_config_dict(bit),
        "dataset_ids": list(dataset_ids),
    }


def save_checkpoint(recs, ds, model, bit, seed, dataset_ids):
    payload = {"meta": _ckpt_meta(ds, model, bit, seed, dataset_ids),
               "records": recs}
    with open(_ckpt_path(ds, model, bit, seed), "w") as f:
        json.dump(payload, f, indent=2)


def load_checkpoint(ds, model, bit, seed, dataset_ids):
    p = _ckpt_path(ds, model, bit, seed)
    if not p.exists():
        return []
    try:
        with open(p) as f:
            payload = json.load(f)
    except Exception as e:
        log.warning(f"  Checkpoint {p.name} unreadable ({e}) - discarding.")
        return []

    expected = _ckpt_meta(ds, model, bit, seed, dataset_ids)
    got_meta = payload.get("meta", {})
    for k, v in expected.items():
        if got_meta.get(k) != v:
            log.warning(f"  Checkpoint {p.name} metadata mismatch on '{k}' - discarding.")
            return []

    saved_records = payload.get("records", [])
    saved_ids = [r.get("item_id") for r in saved_records]
    expected_prefix = list(dataset_ids[:len(saved_ids)])

    if saved_ids != expected_prefix:
        log.warning(
            f"  Checkpoint {p.name} ID sequence does not match the current "
            f"dataset prefix (saved={saved_ids[:5]}... expected={expected_prefix[:5]}...). "
            f"Discarding checkpoint."
        )
        return []

    return saved_records


def save_master(recs, model, bit, seed=None):
    safe_m = model.replace("/", "_")
    prec = {16: "FP16", 8: "8bit", 4: "4bit"}.get(bit, f"{bit}bit")
    tag = f"__seed{seed}" if seed is not None else ""
    fname = f"qp2_master__{safe_m}__{prec}{tag}.json"
    payload = {
        "model": model, "bit_width": bit, "prec_label": prec, "seed": seed,
        "n_records": len(recs),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "factuality_validation_status": FACTUALITY_VALIDATION_STATUS,
        "generation_params": {"max_new_tokens": MAX_NEW_TOKENS,
                              "temperature": TEMPERATURE,
                              "top_p": TOP_P, "top_k": TOP_K,
                              "do_sample": DO_SAMPLE, "num_beams": NUM_BEAMS},
        "bnb_config": _bnb_config_dict(bit),
        "records": recs,
    }
    for d in [CHECKPOINT_DIR, RESULTS_DIR, Path("/kaggle/working")]:
        try:
            with open(d / fname, "w") as f:
                json.dump(payload, f, indent=2)
        except Exception:
            pass
    log.info(f"  Master JSON saved: {fname}")
    return fname


# ---------------------------------------------------------------------------
# Evaluation loop
# ---------------------------------------------------------------------------
def evaluate_dataset(dataset, tok, model, model_name, bit, seed, resume=True):
    ds_name = dataset[0]["dataset"] if dataset else "unknown"
    dataset_ids = [r.get("item_id") for r in dataset]

    results = load_checkpoint(ds_name, model_name, bit, seed, dataset_ids) if resume else []
    if results:
        log.info(f"  Resuming {ds_name} from {len(results)} samples "
                 f"(prefix validated).")

    already = len(results)
    total = len(dataset)

    pbar = tqdm(total=total, initial=already,
                desc=f"{model_name.split('/')[-1]} {bit}bit | {ds_name}",
                unit="sample", dynamic_ncols=True)

    for i, row in enumerate(dataset):
        if i < already:
            continue
        try:
            resp, ent, lat = generate_response(row["prompt"], tok, model)
            lab = classify_heuristic(resp, row.get("reference", ""))
        except Exception as e:
            log.warning(f"  Sample {i} failed: {e}")
            resp, ent, lat, lab = "", 0.0, 0.0, "unscored"

        results.append({
            "index": i,
            "item_id": row.get("item_id"),
            "dataset": ds_name,
            "split": row.get("split", "test"),
            "category": row.get("category", "general"),
            "prompt": row["prompt"][:300],
            "reference": row.get("reference", ""),
            "response": resp[:600],
            "entropy": round(ent, 4),
            "entropy_temperature": TEMPERATURE,
            "entropy_source": "generate_output_scores_full_vocab",
            "confidence_proxy": round(math.exp(-ent), 4),
            "latency_seconds": round(lat, 3),
            "heuristic_label": lab,
            "factuality_validation_status": FACTUALITY_VALIDATION_STATUS,
            "uncertainty_label": "low",
            "theta_used": None,
            "calibration_method": None,
            "in_calibration": False,
            "model": model_name,
            "bit_width": bit,
            "seed": seed,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        })
        if (i + 1) % SAVE_EVERY == 0:
            save_checkpoint(results, ds_name, model_name, bit, seed, dataset_ids)
        pbar.update(1)

    pbar.close()
    save_checkpoint(results, ds_name, model_name, bit, seed, dataset_ids)
    return results


# ---------------------------------------------------------------------------
# Aggregation
# ---------------------------------------------------------------------------
def aggregate(recs, model, bit):
    s = {"model": model, "bit_width": bit,
         "timestamp": datetime.now().isoformat(timespec="seconds"),
         "factuality_validation_status": FACTUALITY_VALIDATION_STATUS,
         "datasets": {}, "overall": compute_metrics(recs)}
    for ds in ["TruthfulQA", "FreshQA", "QuantHall-Bench"]:
        sub = [r for r in recs if r["dataset"] == ds]
        if sub:
            s["datasets"][ds] = compute_metrics(sub)
    return s


# ---------------------------------------------------------------------------
# Excel output (v9: adds MultiSeed_Summary and Per_Seed sheets)
# ---------------------------------------------------------------------------
def write_excel(summaries, extra=None):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = RESULTS_DIR / f"QuantProbe2_{ts}.xlsx"
    rows = []
    for s in summaries:
        m = s["overall"]
        rows.append({
            "Model": s["model"],
            "Precision": {16: "FP16", 8: "8-bit", 4: "4-bit"}[s["bit_width"]],
            "N": m.get("n", 0),
            "N_scored": m.get("n_scored", 0),
            "Accuracy_heuristic": m.get("accuracy"),
            "ErrorRate_heuristic": m.get("error_rate"),
            "HR_proxy": m.get("hallucination_rate_proxy"),
            "OCS_proxy": m.get("overconfidence_proxy"),
            "AR": m.get("abstention_rate"),
            "ECE_proxy": m.get("ece_proxy"),
            "Brier": m.get("brier"),
            "AUROC": m.get("auroc_error"),
            "AURC": m.get("aurc"),
            "CRS_clamped": m.get("crs_clamped"),
            "ErrEntMean": m.get("err_entropy_mean"),
            "CalibMethod": s.get("calibration_method", "unknown"),
            "FactualityValidation": s.get("factuality_validation_status",
                                          FACTUALITY_VALIDATION_STATUS),
        })

    with pd.ExcelWriter(out, engine="openpyxl") as w:
        pd.DataFrame(rows).to_excel(w, sheet_name="Summary", index=False)

        # Multi-seed aggregate (mean and std across seeds for every metric)
        ms_rows = []
        for s in summaries:
            ms = s.get("multi_seed")
            if not ms:
                continue
            row = {"Model": s["model"],
                   "Precision": {16: "FP16", 8: "8-bit", 4: "4-bit"}[s["bit_width"]]}
            for metric, stats in ms.items():
                row[f"{metric}_mean"] = round(stats["mean"], 4)
                row[f"{metric}_std"]  = round(stats["std"], 4)
            ms_rows.append(row)
        if ms_rows:
            pd.DataFrame(ms_rows).to_excel(w, sheet_name="MultiSeed_Summary", index=False)

        # Per-seed rows for traceability
        per_seed_rows = []
        for s in summaries:
            ms = s.get("multi_seed")
            if not ms:
                continue
            prec = {16: "FP16", 8: "8-bit", 4: "4-bit"}[s["bit_width"]]
            for metric, stats in ms.items():
                for seed, val in zip(RANDOM_SEEDS, stats["per_seed"]):
                    per_seed_rows.append({
                        "Model": s["model"], "Precision": prec,
                        "Metric": metric, "Seed": seed, "Value": round(val, 4),
                    })
        if per_seed_rows:
            pd.DataFrame(per_seed_rows).to_excel(w, sheet_name="Per_Seed", index=False)

        bnb = []
        for (m, b) in ALL_EXPERIMENTS:
            for k, v in _bnb_config_dict(b).items():
                bnb.append({"model": m, "bit_width": b, "param": k, "value": str(v)})
        pd.DataFrame(bnb).to_excel(w, sheet_name="BnB_Config", index=False)

        if extra:
            for name, df in extra.items():
                if isinstance(df, pd.DataFrame):
                    df.to_excel(w, sheet_name=name[:31], index=False)
    log.info(f"Excel: {out}")
    return out


# ---------------------------------------------------------------------------
# Seed runner
# ---------------------------------------------------------------------------
def _run_one_seed(model_name, bit, seed, all_records_out):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    tok, model = load_model(model_name, bit)
    n = get_sample_size(bit)

    truthfulqa_data = load_truthfulqa(n)
    freshqa_data    = load_freshqa(n, model_name=model_name)
    bench_dev       = load_quanthallbench(n, split="dev")
    bench_test      = load_quanthallbench(n, split="test")

    eval_data = truthfulqa_data + freshqa_data + bench_test

    recs = []
    for ds in ["TruthfulQA", "FreshQA", "QuantHall-Bench"]:
        sub = [r for r in eval_data if r["dataset"] == ds]
        if sub:
            can_resume = (seed == RANDOM_SEEDS[0] and RUN_MODE == "full")
            recs.extend(evaluate_dataset(sub, tok, model, model_name, bit,
                                         seed, resume=can_resume))

    dev_recs = []
    if bench_dev:
        dev_recs = evaluate_dataset(bench_dev, tok, model, model_name, bit,
                                    seed, resume=False)

    unload_model(model)

    theta, calib_ids, _, method = calibrate_threshold_heldout(dev_recs)

    for r in recs:
        r["theta_used"] = theta
        r["uncertainty_label"] = "low" if r["entropy"] < theta else "high"
        r["calibration_method"] = method
        r["in_calibration"] = False

    metrics = compute_metrics(recs)
    metrics["theta_used"] = theta
    metrics["calibration_method"] = method
    metrics["n_dev_records"] = len(dev_recs)
    metrics["n_eval_records"] = len(recs)

    save_master(recs, model_name, bit, seed=seed)
    save_master(dev_recs, model_name, bit, seed=f"{seed}-dev")
    all_records_out[(model_name, bit, seed)] = recs
    return metrics


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main():
    _run_all_selftests()

    if RUN_MODE == "selftest":
        log.info("RUN_MODE=selftest - stopping after self-tests.")
        return

    if torch.cuda.is_available():
        log.info(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        log.warning("No GPU detected.")

    experiments = get_experiments()
    log.info(f"RUN_MODE={RUN_MODE} | experiments={len(experiments)}")

    all_records = {}
    summaries = []

    for idx, (model_name, bit) in enumerate(experiments, 1):
        log.info(f"\n{'#'*70}\nExperiment {idx}/{len(experiments)}: "
                 f"{model_name} at {bit}-bit\n{'#'*70}")

        seeds_for_this_run = [RANDOM_SEEDS[0]] if RUN_MODE == "smoke" else RANDOM_SEEDS
        seed_metrics = []
        for seed in seeds_for_this_run:
            log.info(f"  --- seed {seed} ---")
            m = _run_one_seed(model_name, bit, seed, all_records)
            m["seed"] = seed
            seed_metrics.append(m)

        numeric_keys = [k for k in seed_metrics[0]
                        if isinstance(seed_metrics[0][k], (int, float)) and k != "seed"]
        agg = {k: {"mean": float(np.mean([m[k] for m in seed_metrics])),
                   "std": float(np.std([m[k] for m in seed_metrics])),
                   "per_seed": [m[k] for m in seed_metrics]}
               for k in numeric_keys}
        log.info(f"  Multi-seed: error_rate={agg['error_rate']['mean']:.3f} "
                 f"+/- {agg['error_rate']['std']:.3f}")

        first_recs = all_records[(model_name, bit, RANDOM_SEEDS[0])]
        s = aggregate(first_recs, model_name.split("/")[-1], bit)
        s["multi_seed"] = agg
        s["calibration_method"] = seed_metrics[0].get("calibration_method", "unknown")
        summaries.append(s)

    extra = {}
    for model_name in {m for m, _ in experiments}:
        fp = all_records.get((model_name, 16, RANDOM_SEEDS[0]))
        if not fp:
            continue
        for bit in (8, 4):
            q = all_records.get((model_name, bit, RANDOM_SEEDS[0]))
            if not q:
                continue
            out = paired_transition_matrix(fp, q)
            extra[f"Trans_{model_name.split('/')[-1]}_{bit}"] = (
                out["matrix"].reset_index().rename(columns={"index": "from"})
            )

    # Threshold sensitivity per (model, precision), primary seed only.
    sens_frames = []
    for (model_name, bit, seed), recs in all_records.items():
        if seed != RANDOM_SEEDS[0]:
            continue
        df = threshold_sensitivity(recs)
        if df.empty:
            continue
        df = df.copy()
        df.insert(0, "model", model_name.split("/")[-1])
        df.insert(1, "bit_width", bit)
        sens_frames.append(df)
    if sens_frames:
        extra["Threshold_Sensitivity"] = pd.concat(sens_frames, ignore_index=True)
        summary_sens = []
        for df in sens_frames:
            for _, row in df.iterrows():
                summary_sens.append({
                    "Model":    row["model"],
                    "Precision": {16: "FP16", 8: "8-bit", 4: "4-bit"}[row["bit_width"]],
                    "Threshold": row["threshold"],
                    "N_scored":  row["n_scored"],
                    "ErrorRate": round(row["error_rate"], 4),
                    "OCS_proxy": round(row["ocs_proxy"], 4),
                    "HR_proxy":  round(row["hr_proxy"], 4),
                })
        extra["Threshold_Sensitivity_Rows"] = pd.DataFrame(summary_sens)

    out = write_excel(summaries, extra=extra)
    log.info(f"\nDone. Excel: {out}")
    log.info("")
    log.info("NOTES:")
    log.info("  - All label fields are 'heuristic_label'. These are NOT human-validated.")
    log.info("  - 'factuality_validation_status' is set to "
             f"'{FACTUALITY_VALIDATION_STATUS}' in every output.")
    log.info("  - The entropy threshold is the median entropy of incorrect answers")
    log.info("    in the development calibration subset. It is a heuristic, not an")
    log.info("    optimal threshold.")
    log.info("  - The following tasks require action outside this script:")
    log.info("      * Human annotation of a representative sample.")
    log.info("      * Live anonymized repository.")
    log.info("      * Paper text updates (terminology, claims, limitations).")


if __name__ == "__main__":
    main()